# Comparativa global de preprocesado EEG-fMRI

Este cuaderno lee los resultados generados por `run_batch_preprocessing.py`. Cada métrica compara la señal completa reconstruida por cada estrategia contra su referencia `Gradient_artifact_corrected`.

In [ ]:
from pathlib import Path
import json, sys
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd

START_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (START_DIRECTORY, *START_DIRECTORY.parents) if (path / 'src').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate the repository root containing src/.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from functions.preprocessing_paths import recording_output_name, subject_output_name

DATASET = 'Dataset1'
PREPROCESSING_ROOT = PROJECT_ROOT / 'data' / 'preprocessing' / DATASET
EVALUATION_ROOT = PROJECT_ROOT / 'data' / 'evaluation' / DATASET
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw' / DATASET / 'Simultaneous_EEG_fMRI' / 'BIDS_dataset_EEG'
STRATEGIES = ['APPEAR', 'BCGGAN', 'AAS_AAS_PCA', 'IVA_OBS_ICA', 'DAR']


In [ ]:
# Consolidar los resúmenes escritos por evaluate_eeg_preprocessing.py.
records = []
for path in EVALUATION_ROOT.glob('*/*_summary_metrics.json'):
    with path.open(encoding='utf-8') as handle:
        item = json.load(handle)
    item['strategy'] = path.parent.name
    # El método tiene el formato ESTRATEGIA_subject001_tarea.
    stem = path.name.removesuffix('_summary_metrics.json')
    label = stem.removeprefix(f"{item['strategy']}_")
    item['recording'] = label
    records.append(item)
metrics = pd.DataFrame(records)
if metrics.empty:
    raise FileNotFoundError('No hay evaluaciones todavía. Ejecuta primero run_batch_preprocessing.py.')
for column in ['preprocessing_seconds', 'evaluation_seconds', 'total_seconds']:
    if column not in metrics:
        metrics[column] = np.nan
display(metrics[['strategy', 'recording', 'n_channels', 'seconds_evaluated', 'mean_pearson_r', 'mean_rmse', 'mean_spectral_relative_error', 'preprocessing_seconds', 'total_seconds']].sort_values(['recording', 'strategy']))


In [ ]:
# Tabla principal: promedios entre todos los registros evaluados.
comparison = (metrics.groupby('strategy')[['mean_pearson_r', 'mean_r2', 'mean_rmse', 'mean_mae', 'mean_reconstruction_snr_db', 'mean_spectral_relative_error', 'mean_psd_correlation', 'preprocessing_seconds', 'total_seconds']]
              .agg(['mean', 'std', 'count'])
              .sort_values(('mean_pearson_r', 'mean'), ascending=False))
display(comparison)


In [ ]:
# Distribución por registro: correlación alta y RMSE bajo son mejores.
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), layout='constrained')
for ax, column, title in zip(axes, ['mean_pearson_r', 'mean_rmse', 'preprocessing_seconds'], ['Correlación temporal', 'RMSE (V)', 'Tiempo de preprocesado (s)']):
    order = [name for name in STRATEGIES if name in metrics.strategy.unique()]
    values = [metrics.loc[metrics.strategy.eq(name), column].dropna() for name in order]
    ax.boxplot(values, tick_labels=order, showmeans=True)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=30)
plt.show()


In [ ]:
# Localizar el .set crudo que corresponde a un resultado .npz.
raw_lookup = {}
for raw_path in RAW_ROOT.glob('sub-*/eeg/*_eeg.set'):
    if 'fmri' not in raw_path.name.lower():
        continue
    subject = raw_path.parents[1].name
    task = raw_path.name.split('_task-', 1)[1].removesuffix('_eeg.set')
    raw_lookup[f'{subject_output_name(subject)}_{recording_output_name(task)}'] = raw_path

available = {}
for strategy in STRATEGIES:
    for result in (PREPROCESSING_ROOT / strategy).glob('*.npz'):
        if not result.stem.endswith(('_pca_analysis', '_ica_analysis')):
            available.setdefault(result.stem, {})[strategy] = result
available = {name: runs for name, runs in available.items() if name in raw_lookup}
if not available:
    raise FileNotFoundError('No hay pares crudo/preprocesado disponibles para visualizar.')
sorted(available)


In [ ]:
# Visualizador MNE completo. Cambia estos dos valores y ejecuta la celda.
RECORDING = sorted(available)[0]
STRATEGY = sorted(available[RECORDING])[0]
raw = mne.io.read_raw_eeglab(raw_lookup[RECORDING], preload=True, verbose='ERROR')
with np.load(available[RECORDING][STRATEGY], allow_pickle=False) as archive:
    cleaned = archive['cleaned_signal']
    cleaned_names = [str(x) for x in archive['channel_names']]
if raw.ch_names != cleaned_names:
    raise ValueError('Los canales del resultado no coinciden con el EEG crudo.')
clean_raw = mne.io.RawArray(cleaned, raw.info.copy(), first_samp=raw.first_samp, verbose='ERROR')
print(f'{RECORDING} | {STRATEGY}: abre dos ventanas MNE para inspección completa.')
raw.plot(title='Antes: EEG crudo', block=False)
clean_raw.plot(title=f'Después: {STRATEGY}', block=False)


In [ ]:
# Comparación ajustable en Matplotlib (canal, inicio y duración).
# Requiere ejecutar antes la celda de MNE.
import ipywidgets as widgets
from IPython.display import display

def plot_segment(channel, start_seconds, duration_seconds):
    fs = raw.info['sfreq']
    start = int(start_seconds * fs)
    stop = min(raw.n_times, start + int(duration_seconds * fs))
    idx = raw.ch_names.index(channel)
    time = np.arange(start, stop) / fs
    fig, ax = plt.subplots(figsize=(14, 4), layout='constrained')
    ax.plot(time, raw.get_data(picks=[idx], start=start, stop=stop)[0] * 1e6, label='Crudo', alpha=.7)
    ax.plot(time, clean_raw.get_data(picks=[idx], start=start, stop=stop)[0] * 1e6, label=STRATEGY, alpha=.8)
    ax.set(xlabel='Tiempo (s)', ylabel='Amplitud (µV)', title=f'{RECORDING} — {channel}')
    ax.legend(); plt.show()

display(widgets.interactive(
    plot_segment,
    channel=widgets.Dropdown(options=raw.ch_names, value=raw.ch_names[0]),
    start_seconds=widgets.FloatSlider(value=0, min=0, max=max(0, raw.times[-1] - 10), step=1, description='Inicio (s)'),
    duration_seconds=widgets.FloatSlider(value=10, min=1, max=60, step=1, description='Duración (s)'),
))


## Análisis guardados

Los métodos `AAS_AAS_PCA`, `APPEAR` e `IVA_OBS_ICA` conservan respectivamente archivos `*_pca_analysis.npz`, `*_ica.fif` y `*_ica_analysis.npz` junto a cada resultado. El archivo `.fif` puede abrirse con MNE para ver mapas y fuentes de ICA; los `.npz` contienen topografías, componentes excluidos y puntuaciones candidatas.